# 02. 메시지와 제공자 전환

같은 메시지를 세 제공자 설정에 적용할 때 무엇을 고정하고 무엇을 기록해야 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 세 provider 통합 패키지의 검증 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더의 .env에서 OpenAI 키 존재 여부만 확인하고 비밀값은 출력하지 않습니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 기본 OpenAI 모델 객체의 모델 ID와 초 단위 제한을 설정합니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## 같은 과제 정의

질문과 출력 길이 요구를 고정합니다.

In [4]:
# provider마다 환경 변수 이름과 model ID를 한 쌍으로 기록합니다.
messages = [
    ("system", "한 문장의 한국어로 답하세요."),
    ("user", "채팅 모델에서 system 메시지의 역할은 무엇인가요?"),
]
providers = {
    "openai": ("OPENAI_API_KEY", "openai:gpt-5.4-mini"),
    "anthropic": ("ANTHROPIC_API_KEY", "anthropic:claude-haiku-4-5"),
    "google": ("GOOGLE_API_KEY", "google_genai:gemini-2.5-flash-lite"),
}
print({name: bool(os.getenv(key)) for name, (key, _) in providers.items()})

{'openai': True, 'anthropic': False, 'google': False}


## 제공자별 실행

설치와 키가 모두 준비된 제공자만 호출합니다. 이번 검증 환경에서는 OpenAI만 실행합니다.

In [5]:
# 키가 있으면 해당 모델을 호출하고, 없으면 continue로 건너뜀 상태를 남깁니다.
records = []
for name, (key_name, model_id) in providers.items():
    if not os.getenv(key_name):  # 키 필요
        records.append({"provider": name, "status": f"skipped: {key_name} 필요"})
        continue
    current = init_chat_model(model_id, timeout=30, max_retries=1, max_tokens=180)
    answer = current.invoke(messages)
    records.append({
        "provider": name, "status": "executed", "text": answer.text,
        "usage": answer.usage_metadata, "content_blocks": answer.content_blocks,
    })
for record in records:
    print(record)

{'provider': 'openai', 'status': 'executed', 'text': 'system 메시지는 채팅 모델의 동작 규칙, 역할, 말투, 안전 제약 같은 상위 지침을 설정해 이후 대화 전반을 안내하는 역할을 합니다.', 'usage': {'input_tokens': 33, 'output_tokens': 43, 'total_tokens': 76, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}, 'content_blocks': [{'type': 'text', 'text': 'system 메시지는 채팅 모델의 동작 규칙, 역할, 말투, 안전 제약 같은 상위 지침을 설정해 이후 대화 전반을 안내하는 역할을 합니다.'}]}
{'provider': 'anthropic', 'status': 'skipped: ANTHROPIC_API_KEY 필요'}
{'provider': 'google', 'status': 'skipped: GOOGLE_API_KEY 필요'}


## 메시지 객체로 같은 뜻 표현

튜플 대신 명시적인 메시지 클래스를 사용해 역할을 확인합니다.

`SystemMessage(...)`와 `HumanMessage(...)`는 전달할 메시지 객체를 생성하며 여기서는 서버를 호출하지 않습니다. 문자열은 각 객체의 내용입니다. 객체의 `.type`은 역할 종류를 표시합니다. 사용자 역할을 튜플로 쓸 때의 `user`가 `HumanMessage.type`에서는 `human`으로 표시됩니다.

번역 규칙을 공통 system 메시지로 두고 여러 번역 대상 문장만 바꿀 수도 있습니다. 객체 형식은 역할과 내용을 분리해 읽기 위한 표현이며, 메시지 클래스 자체가 응답을 더 정확하게 만드는 설정은 아닙니다. 아래 셀은 역할 목록만 출력하므로 튜플과 객체 방식의 모델 응답 품질을 비교한 실험으로 해석하지 않습니다.

In [6]:
# 튜플과 같은 역할 구조를 명시적인 SystemMessage·HumanMessage로 만듭니다.
from langchain.messages import HumanMessage, SystemMessage

objects = [
    SystemMessage("한 문장의 한국어로 답하세요."),
    HumanMessage("채팅 모델에서 system 메시지의 역할은 무엇인가요?"),
]
print([item.type for item in objects])

['system', 'human']


## 실험 결과

실행된 제공자와 건너뛴 제공자를 분리해 집계합니다.

In [7]:
# records에서 실제 실행 행과 키가 없어 건너뛴 행을 분리합니다.
executed = [r["provider"] for r in records if r["status"] == "executed"]
skipped = [r["provider"] for r in records if r["status"].startswith("skipped")]
print("executed:", executed)
print("skipped:", skipped)

executed: ['openai']
skipped: ['anthropic', 'google']
